# arXiv HTML Reference Text Notebook

Notebook này fetch **HTML full-text** từ arXiv cho các bài đã có trong danh sách metadata,
parse và chuẩn hóa ra **plain text sạch** để dùng làm reference text.

**Pipeline:**
1. Load danh sách `arxiv_id` từ `arxiv_ocr_benchmark_workspace/metadata/`
2. Build HTML URL: `https://arxiv.org/html/{arxiv_id}`
3. Fetch HTML → lưu raw HTML
4. Parse HTML → chuẩn hóa plain text
5. Xuất `parsed_jsonl/arxiv_html_reference.jsonl` (cùng schema với Tesseract output)

**Output schema mỗi record:**
```
doc_id, parser_name, source_type, title, primary_category,
published, html_url, pdf_url, char_count, text_sha1, text, fetch_status, metadata
```

> **Ghi chú:** arXiv HTML có sẵn từ ~2023 trở đi cho các bài dùng LaTeX.
> Bài cũ hơn hoặc không phải LaTeX → `fetch_status=failed`, vẫn ghi record để track.

## 0. Cài thư viện

In [10]:
%pip -q install requests beautifulsoup4 lxml tqdm pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


## 1. Import và cấu hình

In [11]:
from pathlib import Path
import re
import json
import time
import random
import hashlib
import concurrent.futures
import threading

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

random.seed(42)

# ===== Input: metadata từ notebook PyMuPDF/Tesseract =====
PREV_META_DIR = Path("arxiv_ocr_benchmark_workspace/metadata")
META_JSONL = PREV_META_DIR / "arxiv_search_results_multi_query.jsonl"
META_CSV   = PREV_META_DIR / "arxiv_search_results_multi_query.csv"

# ===== Output =====
BASE_DIR   = Path("arxiv_html_reference_workspace")
HTML_DIR   = BASE_DIR / "html_raw"      # HTML thô
TEXT_DIR   = BASE_DIR / "html_text"     # Plain text đã parse
PARSED_DIR = BASE_DIR / "parsed_jsonl"  # JSONL tổng hợp
META_DIR   = BASE_DIR / "metadata"

for d in [HTML_DIR, TEXT_DIR, PARSED_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== Fetch config =====
NUM_WORKERS = 4      # số thread song song
SLEEP_SEC   = 1.5    # sleep giữa các request trong cùng thread
MAX_RETRIES = 5      # số lần retry khi lỗi
TIMEOUT     = 30     # timeout mỗi request (giây)
MIN_CHARS   = 300    # bỏ qua record có text ngắn hơn (likely 404)

print(f"Input metadata : {META_JSONL}  exists={META_JSONL.exists()}")
print(f"Output dir     : {BASE_DIR}")
print(f"Workers        : {NUM_WORKERS}  |  sleep={SLEEP_SEC}s  |  retries={MAX_RETRIES}")

Input metadata : arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.jsonl  exists=True
Output dir     : arxiv_html_reference_workspace
Workers        : 4  |  sleep=1.5s  |  retries=5


## 2. Load metadata

In [12]:
if META_JSONL.exists():
    rows = []
    with open(META_JSONL, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} records từ JSONL")
elif META_CSV.exists():
    df = pd.read_csv(META_CSV)
    print(f"Loaded {len(df)} records từ CSV")
else:
    raise FileNotFoundError(
        f"Không tìm thấy metadata.\n"
        f"Hãy chạy notebook arxiv-same-domain-benchmark (hoặc arxiv-tesseract-ocr-benchmark) trước "
        f"để tạo file:\n  {META_JSONL}"
    )

# Build HTML URL từ arxiv_id
def build_html_url(arxiv_id: str) -> str:
    """arXiv HTML URL: bỏ version suffix nếu có thể, thử cả 2 dạng."""
    # Ưu tiên dùng full id kèm version
    return f"https://arxiv.org/html/{arxiv_id}"

df["html_url"] = df["arxiv_id"].apply(build_html_url)

print(f"\nCột: {list(df.columns)}")
print(f"\nSample html_url:")
print(df["html_url"].head(3).to_string())
df[["arxiv_id", "title", "primary_category", "published", "html_url"]].head(5)

Loaded 1579 records từ JSONL

Cột: ['id', 'arxiv_id', 'title', 'summary', 'published', 'updated', 'authors', 'pdf_url', 'primary_category', 'categories', 'requested_query', 'html_url']

Sample html_url:
0    https://arxiv.org/html/2604.01212v1
1    https://arxiv.org/html/2604.01195v1
2    https://arxiv.org/html/2604.01193v1


,arxiv_id,title,primary_category,published,html_url
0,2604.01212v1,$\texttt{YC-Bench}$: Benchmarking AI Agents fo...,cs.CL,2026-04-01T17:52:19Z,https://arxiv.org/html/2604.01212v1
1,2604.01195v1,ORBIT: Scalable and Verifiable Data Generation...,cs.CL,2026-04-01T17:42:41Z,https://arxiv.org/html/2604.01195v1
2,2604.01193v1,Embarrassingly Simple Self-Distillation Improv...,cs.CL,2026-04-01T17:39:50Z,https://arxiv.org/html/2604.01193v1
3,2604.01181v1,True (VIS) Lies: Analyzing How Generative AI R...,cs.HC,2026-04-01T17:33:10Z,https://arxiv.org/html/2604.01181v1
4,2604.01029v1,Revision or Re-Solving? Decomposing Second-Pas...,cs.SE,2026-04-01T15:39:40Z,https://arxiv.org/html/2604.01029v1


## 3. HTML Fetcher và Parser

In [13]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; arxiv-html-crawler/1.0; research-only)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9",
})


def fetch_html(url: str, timeout: int = TIMEOUT, max_retries: int = MAX_RETRIES):
    """Fetch HTML với retry/backoff. Trả (html_text, status_code)."""
    last_status = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = SESSION.get(url, timeout=timeout, allow_redirects=True)
            last_status = resp.status_code
            if resp.status_code == 200:
                return resp.text, 200
            if resp.status_code == 404:
                return None, 404   # không có HTML, không retry
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = (2 ** attempt) + random.uniform(0, 1.0)
                print(f"  [HTTP {resp.status_code}] {url[:80]} → wait {wait:.1f}s ({attempt}/{max_retries})")
                time.sleep(wait)
                continue
            return None, resp.status_code
        except requests.exceptions.Timeout:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"  [Timeout] {url[:80]} → wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"  [Error] {e} → wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
    return None, last_status


# Tags cần xóa hoàn toàn (kể cả nội dung)
_REMOVE_TAGS = [
    "script", "style", "noscript", "nav", "footer", "header",
    "aside", "figure", "figcaption", "svg", "math",
    "button", "form", "input", "select", "textarea",
]

# CSS selectors ưu tiên content chính của arXiv HTML
_CONTENT_SELECTORS = [
    "article",           # arXiv HTML5 paper dùng <article>
    "section.ltx_document",  # LaTeXML output
    "div.ltx_page_content",
    "main",
    "div#content",
    "body",
]


def parse_html_to_text(html: str) -> str:
    """Parse HTML → plain text sạch."""
    soup = BeautifulSoup(html, "lxml")

    for tag_name in _REMOVE_TAGS:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    content_node = None
    for selector in _CONTENT_SELECTORS:
        node = soup.select_one(selector)
        if node and len(node.get_text(strip=True)) > 200:
            content_node = node
            break
    if content_node is None:
        content_node = soup

    text = content_node.get_text(separator="\n")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


print("✅ Fetcher và parser sẵn sàng")
print(f"Content selectors (theo thứ tự ưu tiên): {_CONTENT_SELECTORS}")

✅ Fetcher và parser sẵn sàng
Content selectors (theo thứ tự ưu tiên): ['article', 'section.ltx_document', 'div.ltx_page_content', 'main', 'div#content', 'body']


## 4. Test thử 1 bài

In [14]:
sample = df.iloc[0]
print(f"Doc  : {sample['arxiv_id']}")
print(f"Title: {sample['title'][:80]}")
print(f"URL  : {sample['html_url']}")
print(f"Published: {sample['published'][:10]}")

html_raw, status = fetch_html(sample["html_url"])
if html_raw:
    text = parse_html_to_text(html_raw)
    print(f"\n✅ Fetch OK (HTTP {status}) — HTML: {len(html_raw):,} bytes | text: {len(text):,} chars")
    print("\n--- Text preview (600 chars) ---")
    print(text[:600])
else:
    print(f"\n❌ Fetch thất bại (HTTP {status}) — bài này chưa có HTML trên arXiv")
    print("Thử bài thứ 2...")
    sample2 = df[df["published"] >= "2023-01-01"].iloc[0]
    html_raw2, status2 = fetch_html(sample2["html_url"])
    if html_raw2:
        text2 = parse_html_to_text(html_raw2)
        print(f"✅ Bài '{sample2['arxiv_id']}' — HTTP {status2} | {len(text2):,} chars")
        print(text2[:400])
    else:
        print(f"❌ Bài 2 cũng thất bại (HTTP {status2})")

Doc  : 2604.01212v1
Title: $\texttt{YC-Bench}$: Benchmarking AI Agents for Long-Term Planning and Consisten
URL  : https://arxiv.org/html/2604.01212v1
Published: 2026-04-01

✅ Fetch OK (HTTP 200) — HTML: 279,573 bytes | text: 31,258 chars

--- Text preview (600 chars) ---
\ycbench
: Benchmarking AI Agents for Long-Term Planning and Consistent Execution

Muyu He

  

Adit Jain

  

Anand Kumar

  

Vincent Tu

  

Soumyadeep Bakshi

  

Sachin Patro

  

Nazneen Rajani

(
April 1, 2026
)

Abstract

As LLM agents tackle increasingly complex tasks, a critical question is whether they can maintain strategic coherence over long horizons: planning under uncertainty, learning from delayed feedback, and adapting when early mistakes compound. We introduce
\ycbench
, a benchmark that evaluates these capabilities by tasking an agent with running a simulated startup over a


In [16]:
# Kiểm tra nhanh 5 bài: tỉ lệ HTML available trong dataset
print("=== Kiểm tra nhanh coverage HTML (5 bài mẫu) ===\n")

sample5 = df.head(5)
for _, row in sample5.iterrows():
    html_raw, status = fetch_html(row["html_url"])
    txt_len = len(parse_html_to_text(html_raw)) if html_raw else 0
    icon = "✅" if txt_len >= MIN_CHARS else "❌"
    year = row["published"][:4]
    print(f"{icon} [{year}] {row['arxiv_id']:<20} HTTP={status}  chars={txt_len:>7,}  {row['title'][:50]}")


=== Kiểm tra nhanh coverage HTML (5 bài mẫu) ===

✅ [2026] 2604.01212v1         HTTP=200  chars= 31,258  $\texttt{YC-Bench}$: Benchmarking AI Agents for Lo
❌ [2026] 2604.01195v1         HTTP=404  chars=      0  ORBIT: Scalable and Verifiable Data Generation for
✅ [2026] 2604.01193v1         HTTP=200  chars=106,308  Embarrassingly Simple Self-Distillation Improves C
✅ [2026] 2604.01181v1         HTTP=200  chars= 67,623  True (VIS) Lies: Analyzing How Generative AI Recog
✅ [2026] 2604.01029v1         HTTP=200  chars= 32,828  Revision or Re-Solving? Decomposing Second-Pass Ga


## 5. Fetch và parse toàn bộ

- Resume: bỏ qua `arxiv_id` đã có trong output JSONL
- Ghi incremental sau mỗi record
- Lưu HTML thô vào `html_raw/` và text vào `html_text/`

In [6]:
RESUME_PATH = PARSED_DIR / "arxiv_html_reference.jsonl"
_lock = threading.Lock()

# --- Resume: load các doc_id đã xong ---
done_ids = set()
valid_records = []

if RESUME_PATH.exists():
    with open(RESUME_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                doc = json.loads(line)
                done_ids.add(doc["doc_id"])
                valid_records.append(doc)
            except Exception:
                pass
    # Ghi lại file sạch
    with open(RESUME_PATH, "w", encoding="utf-8") as f:
        for r in valid_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"▶ Resume: {len(done_ids)} records đã xong")

records = list(valid_records)
errors  = []

todo_df = df[~df["arxiv_id"].astype(str).isin(done_ids)].copy()
print(f"Cần fetch: {len(todo_df)} | Bỏ qua (đã xong): {len(done_ids)}")


def process_one(row: dict) -> tuple:
    arxiv_id = str(row.get("arxiv_id", ""))
    html_url = str(row.get("html_url", ""))
    title    = str(row.get("title", ""))
    cat      = str(row.get("primary_category", ""))
    pub      = str(row.get("published", ""))
    pdf_url  = str(row.get("pdf_url", ""))

    safe_id   = re.sub(r"[^\w.\-]", "_", arxiv_id)[:120]
    html_path = HTML_DIR / f"{safe_id}.html"
    text_path = TEXT_DIR / f"{safe_id}.txt"

    html_raw, status = fetch_html(html_url)
    time.sleep(SLEEP_SEC + random.uniform(0, 0.5))

    if html_raw is None:
        rec = {
            "doc_id":       arxiv_id,
            "parser_name":  "arxiv_html",
            "source_type":  "html_reference",
            "title":        title,
            "primary_category": cat,
            "published":    pub,
            "html_url":     html_url,
            "pdf_url":      pdf_url,
            "html_path":    "",
            "text_path":    "",
            "char_count":   0,
            "line_count":   0,
            "text_sha1":    "",
            "text":         "",
            "fetch_status": f"failed_{status}",
            "metadata": {
                "source": "arxiv",
                "http_status": status,
                "html_available": False,
                "summary": str(row.get("summary", "")),
                "authors": row.get("authors", []),
                "categories": row.get("categories", []),
            },
        }
        return rec, None

    # Lưu HTML thô
    html_path.write_text(html_raw, encoding="utf-8", errors="replace")

    # Parse → plain text
    try:
        text = parse_html_to_text(html_raw)
    except Exception as e:
        return None, {"doc_id": arxiv_id, "error": f"parse error: {e}"}

    text_path.write_text(text, encoding="utf-8", errors="replace")

    is_valid = len(text) >= MIN_CHARS
    rec = {
        "doc_id":       arxiv_id,
        "parser_name":  "arxiv_html",
        "source_type":  "html_reference",
        "title":        title,
        "primary_category": cat,
        "published":    pub,
        "html_url":     html_url,
        "pdf_url":      pdf_url,
        "html_path":    str(html_path),
        "text_path":    str(text_path),
        "char_count":   len(text),
        "line_count":   text.count("\n") + 1,
        "text_sha1":    sha1_text(text),
        "text":         text,
        "fetch_status": "ok" if is_valid else "empty",
        "metadata": {
            "source": "arxiv",
            "http_status": 200,
            "html_available": is_valid,
            "summary": str(row.get("summary", "")),
            "authors": row.get("authors", []),
            "categories": row.get("categories", []),
        },
    }
    return rec, None


# --- Chạy song song ---
todo_rows  = todo_df.to_dict(orient="records")
newly_done = 0

out_f = open(RESUME_PATH, "a", encoding="utf-8", errors="replace")
pbar  = tqdm(total=len(todo_rows), desc="Fetch HTML")

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    future_map = {executor.submit(process_one, row): row for row in todo_rows}
    for future in concurrent.futures.as_completed(future_map):
        rec, err = future.result()
        with _lock:
            if rec:
                records.append(rec)
                newly_done += 1
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out_f.flush()
            if err:
                errors.append(err)
        pbar.update(1)

pbar.close()
out_f.close()

ok_count    = sum(1 for r in records if r["fetch_status"] == "ok")
empty_count = sum(1 for r in records if r["fetch_status"] == "empty")
fail_count  = sum(1 for r in records if r["fetch_status"].startswith("failed"))

print(f"\n✅ Tổng: {len(records)}  (resume: {len(done_ids)} | mới: {newly_done})")
print(f"   HTML OK (≥{MIN_CHARS} chars): {ok_count}")
print(f"   HTML rỗng/ngắn:               {empty_count}")
print(f"   Fetch thất bại (404/lỗi):      {fail_count}")
print(f"   Lỗi parse:                     {len(errors)}")
if errors:
    print("\nLỗi đầu:", errors[0])

Cần fetch: 1579 | Bỏ qua (đã xong): 0


Fetch HTML: 100%|██████████| 1579/1579 [13:23<00:00,  1.96it/s]


✅ Tổng: 1579  (resume: 0 | mới: 1579)
   HTML OK (≥300 chars): 1421
   HTML rỗng/ngắn:               1
   Fetch thất bại (404/lỗi):      157
   Lỗi parse:                     0


## 6. Thống kê

In [7]:
stats_df = pd.DataFrame([
    {
        "doc_id":           r["doc_id"],
        "primary_category": r["primary_category"],
        "published":        r["published"][:10] if r["published"] else "",
        "char_count":       r["char_count"],
        "line_count":       r["line_count"],
        "fetch_status":     r["fetch_status"],
    }
    for r in records
])

ok_df = stats_df[stats_df["fetch_status"] == "ok"]

print(f"Tổng records  : {len(stats_df)}")
print(f"\nfetch_status:")
print(stats_df["fetch_status"].value_counts().to_string())
print(f"\nPhân bổ theo category (OK only):")
print(ok_df["primary_category"].value_counts().to_string())
if len(ok_df) > 0:
    print(f"\nThống kê char_count (OK only):")
    print(ok_df["char_count"].describe().to_string())
    print(f"\nnote: arXiv HTML chỉ available cho bài từ ~2023. fail_404 = bài cũ hơn.")

Tổng records  : 1579

fetch_status:
fetch_status
ok            1421
failed_404     157
empty            1

Phân bổ theo category (OK only):
primary_category
cs.CL              403
cs.IR              276
cs.LG              207
cs.AI              191
cs.CV              108
cs.SE               37
cs.RO               33
cs.HC               22
cs.CR               22
cs.DB               15
cs.MA               10
cs.CY                9
stat.ML              9
cs.DL                8
cs.SD                7
quant-ph             6
eess.SY              6
eess.AS              5
math.DS              3
cs.NI                3
q-bio.NC             2
cs.GR                2
cs.SI                2
cs.CE                2
cs.ET                2
cs.DS                2
cs.GT                2
math.OC              2
cs.DC                2
physics.ins-det      2
cs.IT                2
cs.NE                2
cs.OS                1
cs.FL                1
cs.PF                1
q-bio.BM             1
astro-ph.IM    

In [17]:
# Đọc lại từ file JSONL và in thống kê chi tiết
from collections import Counter

out_path = PARSED_DIR / "arxiv_html_reference.jsonl"
if not out_path.exists():
    print("Chưa có output. Chạy cell Fetch ở trên trước.")
else:
    recs = []
    with open(out_path, encoding="utf-8") as f:
        for line in f:
            try:
                recs.append(json.loads(line))
            except Exception:
                pass

    total        = len(recs)
    ok_recs      = [r for r in recs if r["fetch_status"] == "ok"]
    fail_recs    = [r for r in recs if r["fetch_status"].startswith("failed")]
    empty_recs   = [r for r in recs if r["fetch_status"] == "empty"]

    print(f"{'='*55}")
    print(f"  arXiv HTML Reference — Kết quả fetch")
    print(f"{'='*55}")
    print(f"  Tổng records        : {total:>6}")
    print(f"  ✅ OK (≥{MIN_CHARS} chars) : {len(ok_recs):>6}  ({len(ok_recs)/total*100:.1f}%)")
    print(f"  ⬜ Empty/ngắn        : {len(empty_recs):>6}  ({len(empty_recs)/total*100:.1f}%)")
    print(f"  ❌ Failed (404/err)  : {len(fail_recs):>6}  ({len(fail_recs)/total*100:.1f}%)")

    if ok_recs:
        chars = [r["char_count"] for r in ok_recs]
        print(f"\n  char_count (OK only):")
        print(f"    min  = {min(chars):>10,}")
        print(f"    mean = {sum(chars)//len(chars):>10,}")
        print(f"    max  = {max(chars):>10,}")

        print(f"\n  Phân bổ theo category (top 10):")
        for cat, cnt in Counter(r["primary_category"] for r in ok_recs).most_common(10):
            bar = "█" * (cnt * 30 // max(Counter(r["primary_category"] for r in ok_recs).values()))
            print(f"    {cat:<12} {cnt:>4}  {bar}")

    if fail_recs:
        status_counts = Counter(r["fetch_status"] for r in fail_recs)
        print(f"\n  Phân bổ lỗi fetch:")
        for s, c in status_counts.most_common():
            print(f"    {s:<20} {c}")


  arXiv HTML Reference — Kết quả fetch
  Tổng records        :   1579
  ✅ OK (≥300 chars) :   1421  (90.0%)
  ⬜ Empty/ngắn        :      1  (0.1%)
  ❌ Failed (404/err)  :    157  (9.9%)

  char_count (OK only):
    min  =      5,552
    mean =     50,044
    max  =    645,059

  Phân bổ theo category (top 10):
    cs.CL         403  ██████████████████████████████
    cs.IR         276  ████████████████████
    cs.LG         207  ███████████████
    cs.AI         191  ██████████████
    cs.CV         108  ████████
    cs.SE          37  ██
    cs.RO          33  ██
    cs.HC          22  █
    cs.CR          22  █
    cs.DB          15  █

  Phân bổ lỗi fetch:
    failed_404           157


## 7. Xuất metadata CSV / JSONL

In [8]:
META_COLS = [
    "doc_id", "parser_name", "source_type", "title",
    "primary_category", "published", "html_url", "pdf_url",
    "html_path", "text_path", "char_count", "line_count",
    "text_sha1", "fetch_status",
]

meta_rows = [{k: r.get(k, "") for k in META_COLS} for r in records]
meta_df   = pd.DataFrame(meta_rows)

csv_path   = META_DIR / "arxiv_html_reference_meta.csv"
jsonl_path = META_DIR / "arxiv_html_reference_meta.jsonl"

meta_df.to_csv(csv_path, index=False, encoding="utf-8")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in meta_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"CSV  : {csv_path}")
print(f"JSONL: {jsonl_path}")
print(f"Full text JSONL: {RESUME_PATH}")
print(f"\nTotal: {len(meta_df)} records")
meta_df.head(5)

CSV  : arxiv_html_reference_workspace\metadata\arxiv_html_reference_meta.csv
JSONL: arxiv_html_reference_workspace\metadata\arxiv_html_reference_meta.jsonl
Full text JSONL: arxiv_html_reference_workspace\parsed_jsonl\arxiv_html_reference.jsonl

Total: 1579 records


,doc_id,parser_name,source_type,title,primary_category,published,html_url,pdf_url,html_path,text_path,char_count,line_count,text_sha1,fetch_status
0,2604.01212v1,arxiv_html,html_reference,$\texttt{YC-Bench}$: Benchmarking AI Agents fo...,cs.CL,2026-04-01T17:52:19Z,https://arxiv.org/html/2604.01212v1,https://arxiv.org/pdf/2604.01212v1,arxiv_html_reference_workspace\html_raw\2604.0...,arxiv_html_reference_workspace\html_text\2604....,31258,538,8aff7ad9f6603e88bdc0ecd22fe4145fef0be877,ok
1,2604.01195v1,arxiv_html,html_reference,ORBIT: Scalable and Verifiable Data Generation...,cs.CL,2026-04-01T17:42:41Z,https://arxiv.org/html/2604.01195v1,https://arxiv.org/pdf/2604.01195v1,,,0,0,,failed_404
2,2604.01181v1,arxiv_html,html_reference,True (VIS) Lies: Analyzing How Generative AI R...,cs.HC,2026-04-01T17:33:10Z,https://arxiv.org/html/2604.01181v1,https://arxiv.org/pdf/2604.01181v1,arxiv_html_reference_workspace\html_raw\2604.0...,arxiv_html_reference_workspace\html_text\2604....,67623,1761,4f18c9d301caa137ef70c2bd61e0a68fffc19241,ok
3,2604.01193v1,arxiv_html,html_reference,Embarrassingly Simple Self-Distillation Improv...,cs.CL,2026-04-01T17:39:50Z,https://arxiv.org/html/2604.01193v1,https://arxiv.org/pdf/2604.01193v1,arxiv_html_reference_workspace\html_raw\2604.0...,arxiv_html_reference_workspace\html_text\2604....,106308,2659,e7b95c7d3c5736e5ab93fcc065c442d357e3e8c9,ok
4,2604.01029v1,arxiv_html,html_reference,Revision or Re-Solving? Decomposing Second-Pas...,cs.SE,2026-04-01T15:39:40Z,https://arxiv.org/html/2604.01029v1,https://arxiv.org/pdf/2604.01029v1,arxiv_html_reference_workspace\html_raw\2604.0...,arxiv_html_reference_workspace\html_text\2604....,32828,720,b94c84ceabd5ad0ac47b0d80c217b692bb5cfda0,ok


## 8. Preview vài sample

In [9]:
ok_records = [r for r in records if r["fetch_status"] == "ok"]

if not ok_records:
    print("Chưa có record OK nào (có thể tất cả bài đều chưa có HTML trên arXiv).")
else:
    samples = random.sample(ok_records, min(3, len(ok_records)))
    for i, rec in enumerate(samples, 1):
        print(f"{'='*70}")
        print(f"[{i}] {rec['doc_id']}")
        print(f"    Title   : {rec['title'][:80]}")
        print(f"    Category: {rec['primary_category']}  |  Published: {rec['published'][:10]}")
        print(f"    chars={rec['char_count']:,}  lines={rec['line_count']:,}  sha1={rec['text_sha1'][:12]}...")
        print(f"    URL: {rec['html_url']}")
        print(f"\n--- Text preview (600 chars) ---")
        print(rec["text"][:600])
        print()

[1] 2603.10494v1
    Title   : VERI-DPO: Evidence-Aware Alignment for Clinical Summarization via Claim Verifica
    Category: cs.CL  |  Published: 2026-03-11
    chars=40,691  lines=993  sha1=58b4caa823ca...
    URL: https://arxiv.org/html/2603.10494v1

--- Text preview (600 chars) ---
VERI-DPO: Evidence-Aware Alignment for Clinical Summarization via Claim Verification and Direct Preference Optimization

Weixin Liu, BS
1
, Congning Ni, PhD
2
, Qingyuan Song, MEng
1
, Susannah L. Rose, PhD
2
, Christopher Symons, PhD
3
, Murat Kantarcioglu, PhD
4
, Bradley A Malin, PhD
1,2
, Zhijun Yin, PhD
1,2

1
Vanderbilt University, Nashville, TN, USA;

2
Vanderbilt University Medical Center, Nashville, TN, USA

3
Lirio, Knoxville, TN, USA

4
Virginia Tech, Blacksburg, VA, USA

Abstract

Brief Hospital Course (BHC) narratives must be clinically useful yet faithful to fragmented EHR eviden

[2] 2604.01169v1
    Title   : Bridging the Simulation-to-Experiment Gap with Generative Models using Adversari

In [18]:
# In chi tiết 1 bài OK bất kỳ
ok_records = [r for r in records if r["fetch_status"] == "ok"]
rec = random.choice(ok_records)

print(f"doc_id          : {rec['doc_id']}")
print(f"title           : {rec['title']}")
print(f"category        : {rec['primary_category']}")
print(f"published       : {rec['published'][:10]}")
print(f"html_url        : {rec['html_url']}")
print(f"fetch_status    : {rec['fetch_status']}")
print(f"char_count      : {rec['char_count']:,}")
print(f"line_count      : {rec['line_count']:,}")
print(f"text_sha1       : {rec['text_sha1']}")
print(f"html_path       : {rec['html_path']}")
print(f"text_path       : {rec['text_path']}")
print(f"\n{'─'*70}")
print(f"TEXT (2000 chars đầu):")
print(f"{'─'*70}")
print(rec["text"][:2000])


doc_id          : 2603.23149v1
title           : Describe-Then-Act: Proactive Agent Steering via Distilled Language-Action World Models
category        : cs.AI
published       : 2026-03-24
html_url        : https://arxiv.org/html/2603.23149v1
fetch_status    : ok
char_count      : 53,021
line_count      : 1,347
text_sha1       : c94787027e321271f0f2215f090bf812e418bb19
html_path       : arxiv_html_reference_workspace\html_raw\2603.23149v1.html
text_path       : arxiv_html_reference_workspace\html_text\2603.23149v1.txt

──────────────────────────────────────────────────────────────────────
TEXT (2000 chars đầu):
──────────────────────────────────────────────────────────────────────
1
1
institutetext:
Sapienza University of Rome, Italy
2
2
institutetext:
Inria, Univ. Grenoble Alpes, CNRS, LJK

Describe-Then-Act: Proactive Agent Steering via Distilled Language-Action World Models

Massimiliano Pappa

Equal contribution

  

Luca Romani
*

  

Valentino Sacco
*

  

Alessio Palma

  

Stép

## 9. Ghi chú sử dụng

### Output files

```
arxiv_html_reference_workspace/
├── html_raw/          # HTML thô mỗi bài
│   └── <arxiv_id>.html
├── html_text/         # Plain text đã parse
│   └── <arxiv_id>.txt
├── parsed_jsonl/
│   └── arxiv_html_reference.jsonl   # ← main output
└── metadata/
    ├── arxiv_html_reference_meta.csv
    └── arxiv_html_reference_meta.jsonl
```

### Ghép với Tesseract/PyMuPDF output

Cùng `doc_id` (`arxiv_id`), khác `parser_name` → near-duplicate positive pairs:

```python
# Tesseract OCR text  (parser_name="tesseract_ocr")
# arxiv_tesseract_ocr_benchmark_workspace/parsed_jsonl/arxiv_tesseract_ocr_all.jsonl

# HTML reference text (parser_name="arxiv_html")
# arxiv_html_reference_workspace/parsed_jsonl/arxiv_html_reference.jsonl

# Join theo doc_id → label=1 (same paper, different parser)
```

### Lưu ý về coverage
- `fetch_status=ok` → có text dùng được
- `fetch_status=failed_404` → bài chưa có HTML (thường là bài trước 2023 hoặc submission không phải LaTeX)
- Bài từ 2023 trở đi hầu hết có HTML nếu dùng LaTeX